In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
sys.path.append('../src')
from sentiment_utils import (
    get_textblob_score, get_vader_score, get_sentiment_label,
    align_news_to_trading_days, aggregate_daily_sentiment
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
news_df = pd.read_csv('../data/processed/news_processed.csv')
stock_df = pd.read_csv('../data/processed/stock_with_indicators.csv')

# Convert date columns to datetime
news_df['date'] = pd.to_datetime(news_df['date'])
stock_df['Date'] = pd.to_datetime(stock_df['Date'])

print(f"News: {news_df.shape}, Stock: {stock_df.shape}")

In [ ]:
print("Calculating sentiment scores...")
news_df['sentiment_textblob'] = news_df['headline'].apply(get_textblob_score)
news_df['sentiment_vader'] = news_df['headline'].apply(get_vader_score)
news_df['sentiment_category'] = news_df['sentiment_vader'].apply(lambda x: get_sentiment_label(x))

print(news_df['sentiment_category'].value_counts(normalize=True))

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(news_df['sentiment_vader'], bins=50, alpha=0.7, label='VADER')
axes[0].hist(news_df['sentiment_textblob'], bins=50, alpha=0.5, label='TextBlob')
axes[0].set_title('Sentiment Score Distribution')
axes[0].legend()

news_df['sentiment_category'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Sentiment Category (VADER)')
plt.tight_layout()
plt.savefig('../reports/figures/sentiment_distribution.png', dpi=150)
plt.show()

In [ ]:
# Check if news has 'stock' column
if 'stock' not in news_df.columns:
    print("WARNING: No 'stock' column in news. Adding placeholder 'UNKNOWN'.")
    news_df['stock'] = 'UNKNOWN'

news_aligned = align_news_to_trading_days(news_df, stock_df)
print(f"Aligned {len(news_aligned)} articles (dropped {len(news_df) - len(news_aligned)})")

In [ ]:
daily_sentiment = aggregate_daily_sentiment(news_aligned)
print(daily_sentiment.head())

In [ ]:
# Compute daily and next-day returns
stock_df['daily_return'] = stock_df['daily_return']  # already computed in Task 2
stock_df['next_day_return'] = stock_df['daily_return'].shift(-1)
stock_returns = stock_df[['Date', 'Ticker', 'daily_return', 'next_day_return']].copy()
stock_returns.rename(columns={'Ticker': 'stock'}, inplace=True)

In [ ]:
merged = pd.merge(
    daily_sentiment,
    stock_returns,
    left_on=['stock', 'aligned_date'],
    right_on=['stock', 'Date'],
    how='inner'
)
print(f"Merged shape: {merged.shape}")
merged.head()

In [ ]:
# Same-day correlation
corr_vader = merged['sentiment_vader_mean'].corr(merged['daily_return'])
corr_text = merged['sentiment_textblob_mean'].corr(merged['daily_return'])
print(f"VADER vs same-day return: {corr_vader:.4f}")
print(f"TextBlob vs same-day return: {corr_text:.4f}")

# Next-day correlation
corr_vader_next = merged['sentiment_vader_mean'].corr(merged['next_day_return'])
print(f"VADER vs next-day return: {corr_vader_next:.4f}")

# Regression
slope, intercept, r_value, p_value, std_err = stats.linregress(
    merged['sentiment_vader_mean'], merged['daily_return']
)
print(f"Regression R² = {r_value**2:.4f}, p-value = {p_value:.4e}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(merged['sentiment_vader_mean'], merged['daily_return'], alpha=0.5, s=20)

# Regression line
z = np.polyfit(merged['sentiment_vader_mean'], merged['daily_return'], 1)
p = np.poly1d(z)
x_line = np.linspace(merged['sentiment_vader_mean'].min(), merged['sentiment_vader_mean'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', label=f'Corr = {corr_vader:.3f}')

ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Average Daily Sentiment (VADER)')
ax.set_ylabel('Daily Return (%)')
ax.set_title('Sentiment vs Daily Return Correlation')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/sentiment_vs_return_scatter.png', dpi=150)
plt.show()

In [ ]:
cat_returns = merged.groupby('sentiment_category')['daily_return'].agg(['mean', 'std', 'count'])
# Ensure order: Positive, Neutral, Negative
cat_returns = cat_returns.reindex(['Positive', 'Neutral', 'Negative'])

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(cat_returns.index, cat_returns['mean'], yerr=cat_returns['std'], capsize=5, alpha=0.7)
colors = ['green', 'gray', 'red']
for bar, col in zip(bars, colors):
    bar.set_color(col)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Daily Return by News Sentiment Category')
ax.set_ylabel('Average Return (%)')
for i, (idx, row) in enumerate(cat_returns.iterrows()):
    ax.text(i, row['mean'] + (0.1 if row['mean'] >= 0 else -0.3), 
             f"{row['mean']:.3f}%\n(n={int(row['count'])})", 
             ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/sentiment_category_returns.png', dpi=150)
plt.show()

In [ ]:
if len(merged['stock'].unique()) > 1:
    per_stock = merged.groupby('stock').apply(
        lambda x: x['sentiment_vader_mean'].corr(x['daily_return'])
    ).sort_values(ascending=False)
    print("Per-stock correlation (VADER vs daily return):")
    print(per_stock)
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    per_stock.plot(kind='bar', ax=ax)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title('Sentiment-Return Correlation by Stock')
    ax.set_ylabel('Pearson Correlation')
    plt.tight_layout()
    plt.savefig('../reports/figures/per_stock_correlation.png', dpi=150)

In [ ]:
merged.to_csv('../data/processed/sentiment_returns_merged.csv', index=False)
print("Saved merged dataset.")